# Expoloring in SAR in GEE

## Introduction

This notebook is designed to explore SAR data from Sentinel and ALOS PALSAR for the guyana coastline and seek to understand the potential incorporation of mRFDI and other backscatter indices into ML/DL models for extent map classification. The resources helped to assist in this notebook include: 
- [Youtube Tutorial](https://www.youtube.com/watch?v=D3leaw4hcjM&list=PLmk0fUBXB9t0AZioQipqHv5tuE3kpIdvI&index=14)
- [Google Earth Engine Docs](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2#colab-python)
- [Refind Lee Filter Script](https://github.com/ada-zam25/gee_s1_ard/tree/main)
- [Gemini 3.5 Flash](https://gemini.google.com/) 

In [6]:
import ee
import geemap
import matplotlib.pyplot as plt 

ee.Authenticate()
ee.Initialize(project='irp-zu')

### Here we are exploring the polarization bands for Sentinel 1 and mapping it for Guyana's bimodal climate. 

In [7]:
coastline_coords = [[-59.755689785228235, 8.126814578766439],
          [-57.833082363353235, 5.607387745118143],
          [-57.146881951761834, 5.898364580242008],
          [-57.009580630573, 6.02949062794878],
          [-56.888703240824334, 6.15511611857417],
          [-56.493195428324334, 6.624592088692309],
          [-56.416291131449334, 7.785376639618971],
          [-58.45482139063248, 9.543371065466095],
          [-59.98741416406998, 8.545208575574712]]

aoi_polygon = ee.Geometry.Polygon([coastline_coords])

def mask_edge(image):
  edge = image.lt(-30.0)
  masked_image = image.mask().And(edge.Not())
  return image.updateMask(masked_image)


img_vv = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi_polygon)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select('VV')
    .map(mask_edge)
)

desc = img_vv.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
asc = img_vv.filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

wet = ee.Filter.date('2020-01-01', '2020-02-28')
dry = ee.Filter.date('2020-03-01', '2020-03-31')
wet2 = ee.Filter.date('2020-04-01', '2020-08-31')
dry2 = ee.Filter.date('2020-09-01', '2020-11-30')
wet3 = ee.Filter.date('2020-12-01', '2020-12-31')

desc_change = ee.Image.cat(
    desc.filter(wet).mean(),
    desc.filter(wet2).mean(),
    desc.filter(wet3).mean(),
    desc.filter(dry).mean(),
    desc.filter(dry2).mean(),
)

asc_change = ee.Image.cat(
    asc.filter(wet).mean(),
    asc.filter(wet2).mean(),
    asc.filter(wet3).mean(),
    asc.filter(dry).mean(),
    asc.filter(dry2).mean(),
)

m = geemap.Map()
m.center_object(aoi_polygon, 6)
m.add_layer(asc_change, {'min': -25, 'max': 5}, 'Multi-T Mean ASC', True)
m.add_layer(desc_change, {'min': -25, 'max': 5}, 'Multi-T Mean DESC', True)
m

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

In [8]:
# Tutorial: https://www.youtube.com/watch?v=D3leaw4hcjM&list=PLmk0fUBXB9t0AZioQipqHv5tuE3kpIdvI&index=14

s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi_polygon)
    .filterDate('2020-01-01', '2021-01-01')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .select(['VV', 'VH'])
    .map(mask_edge)
)

s1_desc = s1.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
s1_asc = s1.filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))


Map = geemap.Map()
Map.center_object(aoi_polygon, 6)
Map.add_layer(s1_asc, {'min': -25, 'max': 5}, 'Multi-T Mean ASC', True)
Map.add_layer(s1_desc, {'min': -25, 'max': 5}, 'Multi-T Mean DESC', True)
Map

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

### Calculating modified Rainforest Degradation Index for ascending and descending passes for copolar and crosspolar backscatter coefficients

In [9]:
def calculate_mRFDI(image): 
    mRFDI = image.expression('(VV - VH) / (VV + VH)', {
        'VV': image.select('VV'),
        'VH': image.select('VH'),
    }).rename('mRFDI')
    return image.addBands(mRFDI)

s1_with_mRFDI_asc = s1_asc.map(calculate_mRFDI)
s1_with_mRFDI_desc = s1_desc.map(calculate_mRFDI)
Map_RFDI = geemap.Map()
Map_RFDI.center_object(aoi_polygon, 6)
Map_RFDI.add_layer(s1_with_mRFDI_asc, {'min': -25, 'max': 5}, 'mRFI_asc', True)
Map_RFDI.add_layer(s1_with_mRFDI_desc, {'min': -25, 'max': 5}, 'mRFI_dsc', True)
Map_RFDI
# s1_with_mRFDI

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

In [11]:
# Reduce the image to find the actual min and max values in your AOI
# Gemini helped here
stats = s1_with_mRFDI_asc.first().reduceRegion(
    reducer=geemap.ee.Reducer.percentile([2, 98]), # Gets the 2nd and 98th percentile to eliminate outliers
    geometry=aoi_polygon,
    scale=30, # Sentinel-1 resolution is ~10m, but 30m-100m is faster for checking stats
    maxPixels=1e9
)

print(stats.getInfo())

{'VH_p2': -29.61969073822122, 'VH_p98': -13.377025513597617, 'VV_p2': -18.870440176822274, 'VV_p98': -7.377379993843047, 'mRFDI_p2': -0.3000069811611293, 'mRFDI_p98': -0.3000069811611293}


### Applying Refined Lee Filtering per Chan-Bagot et al., 2024's methodology using Mullissa et al., 2021

In [12]:
# https://github.com/ada-zam25/gee_s1_ard/blob/39207cb4998c9c80c27af3d8a62debe7ab41e7c7/python-api/speckle_filter.py#L160

def RefinedLee(image):
    """
    This filter is modified from the implementation by Guido Lemoine 
    Source: Lemoine et al. https://code.earthengine.google.com/5d1ed0a0f0417f098fdfd2fa137c3d0c

    Parameters
    ----------
    image: ee.Image
        Image to be filtered

    Returns
    -------
    result: ee.Image
        Filtered Image

    """

    bandNames = image.bandNames().remove('angle')

    def inner(b):

        img = image.select([b]);
    
        # img must be linear, i.e. not in dB!
        # Set up 3x3 kernels 
        weights3 = ee.List.repeat(ee.List.repeat(1,3),3);
        kernel3 = ee.Kernel.fixed(3,3, weights3, 1, 1, False);
  
        mean3 = img.reduceNeighborhood(ee.Reducer.mean(), kernel3);
        variance3 = img.reduceNeighborhood(ee.Reducer.variance(), kernel3);
  
        # Use a sample of the 3x3 windows inside a 7x7 windows to determine gradients and directions
        sample_weights = ee.List([[0,0,0,0,0,0,0], [0,1,0,1,0,1,0],[0,0,0,0,0,0,0], [0,1,0,1,0,1,0], [0,0,0,0,0,0,0], [0,1,0,1,0,1,0],[0,0,0,0,0,0,0]]);
  
        sample_kernel = ee.Kernel.fixed(7,7, sample_weights, 3,3, False);
  
        # Calculate mean and variance for the sampled windows and store as 9 bands
        sample_mean = mean3.neighborhoodToBands(sample_kernel); 
        sample_var = variance3.neighborhoodToBands(sample_kernel);
  
        # Determine the 4 gradients for the sampled windows
        gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs();
        gradients = gradients.addBands(sample_mean.select(6).subtract(sample_mean.select(2)).abs());
        gradients = gradients.addBands(sample_mean.select(3).subtract(sample_mean.select(5)).abs());
        gradients = gradients.addBands(sample_mean.select(0).subtract(sample_mean.select(8)).abs());
  
        # And find the maximum gradient amongst gradient bands
        max_gradient = gradients.reduce(ee.Reducer.max());
  
        # Create a mask for band pixels that are the maximum gradient
        gradmask = gradients.eq(max_gradient);
  
        # duplicate gradmask bands: each gradient represents 2 directions
        gradmask = gradmask.addBands(gradmask);
  
        # Determine the 8 directions
        directions = sample_mean.select(1).subtract(sample_mean.select(4)).gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1);
        directions = directions.addBands(sample_mean.select(6).subtract(sample_mean.select(4)).gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2));
        directions = directions.addBands(sample_mean.select(3).subtract(sample_mean.select(4)).gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3));
        directions = directions.addBands(sample_mean.select(0).subtract(sample_mean.select(4)).gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4));
        # The next 4 are the not() of the previous 4
        directions = directions.addBands(directions.select(0).Not().multiply(5));
        directions = directions.addBands(directions.select(1).Not().multiply(6));
        directions = directions.addBands(directions.select(2).Not().multiply(7));
        directions = directions.addBands(directions.select(3).Not().multiply(8));
  
        # Mask all values that are not 1-8
        directions = directions.updateMask(gradmask);
  
        # "collapse" the stack into a singe band image (due to masking, each pixel has just one value (1-8) in it's directional band, and is otherwise masked)
        directions = directions.reduce(ee.Reducer.sum());  
  
        sample_stats = sample_var.divide(sample_mean.multiply(sample_mean));
  
        #Calculate localNoiseVariance
        sigmaV = sample_stats.toArray().arraySort().arraySlice(0,0,5).arrayReduce(ee.Reducer.mean(), [0]);
  
        # Set up the 7*7 kernels for directional statistics
        rect_weights = ee.List.repeat(ee.List.repeat(0,7),3).cat(ee.List.repeat(ee.List.repeat(1,7),4));
  
        diag_weights = ee.List([[1,0,0,0,0,0,0], [1,1,0,0,0,0,0], [1,1,1,0,0,0,0], [1,1,1,1,0,0,0], [1,1,1,1,1,0,0], [1,1,1,1,1,1,0], [1,1,1,1,1,1,1]]);
  
        rect_kernel = ee.Kernel.fixed(7,7, rect_weights, 3, 3, False);
        diag_kernel = ee.Kernel.fixed(7,7, diag_weights, 3, 3, False);
  
        # Create stacks for mean and variance using the original kernels. Mask with relevant direction.
        dir_mean = img.reduceNeighborhood(ee.Reducer.mean(), rect_kernel).updateMask(directions.eq(1));
        dir_var = img.reduceNeighborhood(ee.Reducer.variance(), rect_kernel).updateMask(directions.eq(1));
  
        dir_mean = dir_mean.addBands(img.reduceNeighborhood(ee.Reducer.mean(), diag_kernel).updateMask(directions.eq(2)));
        dir_var = dir_var.addBands(img.reduceNeighborhood(ee.Reducer.variance(), diag_kernel).updateMask(directions.eq(2)));
  
        # and add the bands for rotated kernels
        for i in range(1, 4):
            dir_mean = dir_mean.addBands(img.reduceNeighborhood(ee.Reducer.mean(), rect_kernel.rotate(i)).updateMask(directions.eq(2*i+1)))
            dir_var = dir_var.addBands(img.reduceNeighborhood(ee.Reducer.variance(), rect_kernel.rotate(i)).updateMask(directions.eq(2*i+1)))
            dir_mean = dir_mean.addBands(img.reduceNeighborhood(ee.Reducer.mean(), diag_kernel.rotate(i)).updateMask(directions.eq(2*i+2)))
            dir_var = dir_var.addBands(img.reduceNeighborhood(ee.Reducer.variance(), diag_kernel.rotate(i)).updateMask(directions.eq(2*i+2)))

  
        # "collapse" the stack into a single band image (due to masking, each pixel has just one value in it's directional band, and is otherwise masked)
        dir_mean = dir_mean.reduce(ee.Reducer.sum());
        dir_var = dir_var.reduce(ee.Reducer.sum());
  
        # A finally generate the filtered value
        varX = dir_var.subtract(dir_mean.multiply(dir_mean).multiply(sigmaV)).divide(sigmaV.add(1.0))
  
        b = varX.divide(dir_var)
        result = dir_mean.add(b.multiply(img.subtract(dir_mean)))
  
        return result.arrayProject([0]).arrayFlatten([['sum']]).float()
    
    result = ee.ImageCollection(bandNames.map(inner)).toBands().rename(bandNames).copyProperties(image)
    
    return image.addBands(result, None, True) 

### Comparing filtered and non-filtered versions

When building my models, I'll feed the raw and processed bands and run a feature importance experiment to see which performs better. 

In [13]:
s1_default = (
    ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT')
    .filterBounds(aoi_polygon)
    .filterDate('2020-01-01', '2021-01-01')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .select(['VV', 'VH', 'angle'])
    .map(mask_edge)
)

s1_filtered = s1_default.map(RefinedLee)

# Helper function to convert linear backscatter to decibels (dB)
# Formula: dB = 10 * log10(linear)
def to_db(image):
    return image.log10().multiply(10)
    
s1_desc = s1_default.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
s1_asc = s1_default.filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

s1_asc_filtered = s1_filtered.filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
s1_desc_filtered = s1_filtered.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))

s1_asc_median_linear_filtered = s1_asc_filtered.median().select(['VV', 'VH'], ['VV_asc', 'VH_asc'])
s1_desc_median_linear_filtered = s1_desc_filtered.median().select(['VV', 'VH'], ['VV_desc', 'VH_desc'])


s1_asc_median_linear = s1_asc.median().select(['VV', 'VH'], ['VV_asc', 'VH_asc'])
s1_desc_median_linear = s1_desc.median().select(['VV', 'VH'], ['VV_desc', 'VH_desc'])

s1_asc_median_db = to_db(s1_asc_median_linear)
s1_desc_median_db = to_db(s1_desc_median_linear)

s1_asc_filtered_db = to_db(s1_asc_median_linear_filtered)
s1_desc_filtered_db = to_db(s1_desc_median_linear_filtered)

# 1. Create an RGB visualization dictionary using your specific band names
viz_asc_rgb = {
    'min': [-20, -25, 0],   # Min values for [Red, Green, Blue]
    'max': [-5, -10, 15],   # Max values for [Red, Green, Blue]
    'bands': ['VV_asc', 'VH_asc', 'VV_asc'] # Temporarily using VV for Blue just to display an RGB
}

viz_desc_rgb = {
    'min': [-20, -25, 0],
    'max': [-5, -10, 15],
    'bands': ['VV_desc', 'VH_desc', 'VV_desc']
}

M = geemap.Map()
M.center_object(aoi_polygon, 6)
M.add_layer(s1_asc_median_db, viz_asc_rgb, 'S1 ASC', True)
M.add_layer(s1_desc_median_db, viz_desc_rgb, 'S1 DESC', True)

M.add_layer(s1_asc_filtered_db, viz_asc_rgb, 'FILTERED S1 ASC', True)
M.add_layer(s1_desc_filtered_db, viz_desc_rgb, 'FILTERED S1 DESC', True)
M

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

### We'll now look at ALOS PALSAR L-band applying the same preprocessing

In [31]:
# gemini was helpful in this chat: https://gemini.google.com/app/db213973262a323c


m = geemap.Map()

palsar = (
    ee.ImageCollection("JAXA/ALOS/PALSAR/YEARLY/SAR_EPOCH")
        .filterBounds(aoi_polygon)
        .filterDate('2020-01-01', '2021-01-01')
)
palsar_image = palsar.first().clip(aoi_polygon)

sarHh = palsar_image.select('HH')
sarHv = palsar_image.select('HV')

# Go straight from raw DN to Linear Power
linear_sarHh = sarHh.pow(2).multiply(10**(-8.3))
linear_sarHv = sarHv.pow(2).multiply(10**(-8.3))


# Apply Refined Lee Filter
refined_linearHh = RefinedLee(linear_sarHh)
refined_linearHv = RefinedLee(linear_sarHv)

# Example of converting back to dB after filtering
filtered_Hh = refined_linearHh.log10().multiply(10)
filtered_Hv = refined_linearHv.log10().multiply(10)

sarVis = {
  "min": 0.0,
  "max": 10000.0
}

sarVis_dB = {
  "min": -25.0,
  "max": 0.0
}
m.center_object(aoi_polygon, 6);
m.addLayer(sarHh, sarVis, 'SAR HH')
m.addLayer(sarHv, sarVis, 'SAR HV')
m.addLayer(filtered_Hh, sarVis_dB, 'FILTERED SAR HH')
m.addLayer(filtered_Hv, sarVis_dB, 'FILTERED SAR HV')
m

Map(center=[7.625000281267503, -58.02523831094688], controls=(WidgetControl(options=['position', 'transparent_…

In [36]:
sarHh_10m = sarHh.resample('bilinear')
sarHv_10m = sarHv.resample('bilinear')

m.addLayer(sarHh_10m, sarVis, 'SAR HH 10m')
m.addLayer(sarHv_10m, sarVis, 'SAR HV 10m')

print(sarHh_10m.projection().nominalScale().getInfo(), sarHv_10m.projection().nominalScale().getInfo()) # doesn't modify the source file, but you can see the difference

24.73766462072746 24.73766462072746
